## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ~/code/entityrepresentations

In [ ]:
import torch, gc, sys, os, pathlib
import torch.nn as nn
import torch
import numpy as np
from jaxtyping import Float
from torch.utils.data import DataLoader
from datasets import load_dataset
import transformer_lens as tl 
from transformer_lens import HookedTransformer
from importlib import reload
from tqdm import tqdm
import matplotlib.pyplot as plt

#our own code
import utils
reload(utils)
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
from LabelExtractor import eval_model, evaluateTV, infer_entities
from processResults import *


## Model Loading

In [ ]:
# del model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Load a model (eg GPT-2 Small)
model_name = "bloom-3b" #crashed
model_name = "meta-llama/Meta-Llama-3-8B" # ?
model_name = "gpt2-small"   # 117M ok
model_name = "gpt2-xl"      # 1.5B ok
model_name = "gpt2-large"   # 774M ok
model_name = "mistralai/Mistral-7B-v0.1" # ok JZ a100 8cpus | 
model_name = "gpt2-medium" # 302M ok
model_name = "EleutherAI/pythia-6.9b"#? CRASHED
model_name = "EleutherAI/pythia-1.4b"# ok ! (6Gb)
model_name = "EleutherAI/pythia-2.8b"#ok !
model_name = "EleutherAI/pythia-1b"# ok ! (6Gb)
model_name = "EleutherAI/pythia-1b"# ok ! (6Gb)
model_name = "phi-3" # 3,6B -> OK ?
model_name = "pythia-1.4b-deduped-v0"  # 1.5B ok
model_name = "meta-llama/Meta-Llama-3-8B"  # 1.5B ok
model_name = "EleutherAI/pythia-410m"# ok !
model_name = "phi-1_5"  # 1.5B ok
model_name = "phi-2" # 2,5B ok

hf_token= None # your huggingface token if you have one, needed for some models e;g Llama

dtype = torch.float16
dtype = torch.bfloat16
dtype = torch.float32

#check if model variable exists
if not 'model' in locals():
    model = utils.load_llm(model_name, dtype=dtype, token=hf_token)
    print(model)
    model.eval()

model = model.cuda()

with torch.no_grad(): 
    dim = model.QK.shape[-1]
    # dim = model.QK.shape[-1]

In [ ]:
# print hooks
test_prompt = "The quick brown fox jumped over the lazy dog"
print("Num tokens:", len(model.to_tokens(test_prompt)[0]))
def print_name_shape_hook_function(activation, hook):
    print(hook.name, activation.shape)
not_in_late_block_filter = lambda name: name.startswith("blocks.0.") or not name.startswith("blocks")

model.run_with_hooks(
    test_prompt,
    return_type=None,
    fwd_hooks=[(not_in_late_block_filter, print_name_shape_hook_function)],
)

In [ ]:
#test if we get the right hook name
replace_hook_name = tl.utils.get_act_name('embed')
print(replace_hook_name)

In [ ]:
# gt unembed
print(model.W_U.dtype)

In [ ]:
prompt, tok = "When Bob and France went to the store, Bob asked"            , 3
prompt, tok = "When Mary and Bob went to the store, Bob asked"              , 3
prompt, tok = "the French Iron King, aka '"                                 , 4
prompt, tok = "Alan Bean was an ", 2
prompt, tok = "Barack Obama was an ", 3
prompt, tok = "the molecule HU765DGB introduced herself: 'hi, my name is"   , 7
prompt, tok = "The Empire State Building is a great monument"               , 4

prompt, tok = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of \
Texas at Austin in 1955 and was chosen by NASA in 1963. Alan Bean was a"   , 44
prompt, tok = "Man of steel, commonly known as"                             , 3
prompt, tok = "The Eiffel Tower is also known as the 'metal lady' in France."                                           , 5

### Test Generation

In [ ]:
res = model.generate(prompt, max_new_tokens=20, temperature=0, return_type="str")
print(res)
# model.to_str_tokens(res)


### Extract representation at given layer

In [ ]:
toks = model.to_str_tokens(prompt)
for i,t  in enumerate(toks): print(i, "\t", t, "  <--" if i==tok else "")

repr = utils.get_representation(model,
                                layer=9,
                                tokens=model.to_tokens(prompt),
                                token_inds=[tok],
                                verbose=True)

### Regererate Label from rep
Here we can observe that the last subject's token contain enough information to identify the subject.

In [ ]:
utils.generate_from_repr(model, 
                         repr, 
                         retr_prompt="_ is commonly known as", 
                         do_sample=True,)

# Datasets

## WebNLG

### Import

In [ ]:


# Load WebNLG dataset
dataset = load_dataset("web_nlg", "release_v3.0_en", trust_remote_code=True)

#optionnal, filter categories from datset
cat = ['Food'] #Categories to ignore
cat = None
if cat :
    dataset["train"] = [item for item in dataset["train"] if item["category"] not in cat]
    dataset["test"] = [item for item in dataset["test"] if item["category"] not in cat]

# Create dataset instances
train_dataset = utils.WebNLGDataset(dataset['train'], max_ent_length=20)
test_dataset = utils.WebNLGDataset(dataset['test'])


In [ ]:
print("train length:", len(train_dataset))
print("test length:", len(test_dataset))
print("ex sample:", train_dataset[np.random.randint(len(train_dataset))])


### Explore Dataset

In [ ]:
N_train = len(dataset["train"])
print("training dataset: ",N_train)
print("random sample: ")
item = dataset["train"][np.random.randint(N_train)]
# print(item["text"])
item

### Augment Dataset with subject representations

In [ ]:
layer = 15 #layer where to extract representations
b_size = 50
# print("Augmenting Train set with subject representations ... ")
# train_dataset.augment_with_repr(model, layer, batch_size=b_size)
print("Augmenting Test set with subject representations ... ")
test_dataset.augment_with_repr(model, layer, batch_size=b_size)
print("Extraction of subjects representatons Done !\n")

In [ ]:
print(str(train_dataset[torch.randint(len(train_dataset),(1,))]).replace(", '", ",\n'"))

In [ ]:
#ex sample from final dataset
test_dataset[67]["entity"]

### Inspect Dataset

In [ ]:
import numpy as np
# get the first tokens of all entities
first_tokens = []
for i in range(len(train_dataset)):
    first_tokens.append(model.to_str_tokens(train_dataset[i]["entity"])[1])

# get counts of each token
unique, counts = np.unique(first_tokens, return_counts=True)

# sort by counts
sorted_tokens = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)
sorted_tokens[:30]

## TACRED

### Loading

In [ ]:
ds = load_dataset("AmirLayegh/tacred_text_label")
dataset_name = "TACRED"
i = np.random.randint(len(ds["train"]))
print("raw row example:", ds["train"][i])

train_dataset = utils.TacredDataset(ds["train"], max_ent_length=20,max_length=200)
test_dataset = utils.TacredDataset(ds["test"], max_ent_length=20,max_length=200)

#ex sample from final dataset
print("train length:", len(train_dataset))
print("test length:", len(test_dataset))
train_dataset[np.random.randint(len(train_dataset))]

### Augmenting dataset with representations

In [ ]:
i  =np.random.randint(len(train_dataset))
pr = train_dataset[i]["text"].replace("  ", " ")
print(pr)
model.to_str_tokens(pr)

In [ ]:
layer = 15 #layer where to extract representations
b_size = 100
# print("Augmenting Train set with subject representations ... ")
# train_dataset.augment_with_repr(model, layer, batch_size=b_size)
print("Augmenting Test set with subject representations ... ")
test_dataset.augment_with_repr(model, layer, batch_size=b_size)

### Check if Hf version has same data as official
we check in the [json sample](https://catalog.ldc.upenn.edu/desc/addenda/LDC2018T24.json) if we find the same sentences.

In [ ]:
text = "Jeffrey White" # found 
text = "ranked 119"
text = "Douglas Flint" #found 
# search for the subject in the dataset
for split in ["train", "validation", "test"]:
    for i, item in enumerate(ds[split]):
        if item["text"].find(text) != -1:
            print(i, item)

## CoNLL 2003

In [ ]:
from datasets import load_dataset
import utils
ds = load_dataset("eriktks/conll2003", trust_remote_code=True)
dataset_name = "CoNLL2003"
print("Original Dataset:")
print(" - train :", len(ds["train"]))
print(" - test :", len(ds["test"]))
print(" - validation :", len(ds["validation"]))

# Parameters set in LearnLabelExtractor Task
max_ent_length = 60
max_length = 300

train_dataset = utils.CoNLLDataset(
    ds["train"], max_ent_length=max_ent_length, max_length=max_length
)
test_dataset = utils.CoNLLDataset(
    ds["test"], max_ent_length=max_ent_length, max_length=max_length
)
val_dataset = utils.CoNLLDataset(
    ds["validation"], max_ent_length=max_ent_length, max_length=max_length
)
val_dataset.data = val_dataset.data[:1000]  # limit validation set to 1000 samples

# ex sample from final dataset
print("Processed Dataset:")
print(" - train length:", len(train_dataset))
print(" - test length:", len(test_dataset))
print(" - val length:", len(val_dataset))
train_dataset[np.random.randint(len(train_dataset))]

In [ ]:
layer = 20  # layer where to extract representations

# for data in [train_dataset, test_dataset, val_dataset]:
for data in [test_dataset]:
    data.augment_with_repr(model, layer, batch_size=5, 
    method="in_context"
    )

### Exploring Data

In [ ]:
#data keys 
print("train keys:", train_dataset[0].keys())

get_all_unique_ents = lambda data: set([item["entity"] for item in data])

all_ents_train = get_all_unique_ents(train_dataset)
all_ents_test = get_all_unique_ents(test_dataset)

print(f"Number of unique entities in \n - train set: {len(set(all_ents_train))} \n - test set: {len(set(all_ents_test))}")
print()

# gte intersection of entities
intersection = set(all_ents_train).intersection(set(all_ents_test))
percent_in_test = len(intersection) / len(all_ents_test) * 100
print("Number of entities in both sets:", len(intersection))
print("Percentage of entities in test set that are also in train set:", percent_in_test)

print("Example of entity in both sets:", list(intersection)[:15])


In [ ]:
#explore raw data
i = np.random.randint(len(ds["train"]))
item = ds["train"][i]
text = '|'.join(map(lambda x: x.center(10), item["tokens"]))
ner = '|'.join(map(lambda x: str(x).center(10), item["ner_tags"]))
print(text + "\n" + ner)

In [ ]:
# Compute mean token length in dataset
for data in [train_dataset, test_dataset, val_dataset]:
    mean_token_length = np.mean([len(model.to_str_tokens(item['text'])) for item in data])
    print(f'Mean token length in dataset: {mean_token_length:.2f}')

## Baseline, average representation of the entity

In [ ]:
data = [
    {"text":"Have you seen the Eiffel Tower", "entity": "Eiffel Tower", "id": 0},
    {"text": "The guy who won at Waterloo was Napoleon Bonaparte", "entity": "Napoleon Bonaparte", "id": 1},
    {"text": "The capital of France", "entity": "France", "id": 2},        
        ]
    
dataset = utils.EntityReprDataset(data)
dataset.augment_with_avg_repr(model, layer=15, batch_size=3, method="in_context")

In [ ]:
rep = dataset[0]["representation"]
print(rep.shape)

## Baseline, Random spans extraction

In [ ]:
print(len(train_dataset))
n = 1
layer = 10
train_dataset = utils.sample_random_entities(model, train_dataset, n=n)
train_dataset.augment_with_repr(model, layer, batch_size=15, method='in_context')

test_dataset = utils.sample_random_entities(model, test_dataset, n=n)
test_dataset.augment_with_repr(model, layer, batch_size=15, method='in_context')

val_dataset = utils.sample_random_entities(model, val_dataset, n=n)
val_dataset.augment_with_repr(model, layer, batch_size=15, method='in_context')


In [ ]:
print(f"new data length : {len(train_dataset)}")
val_dataset[np.random.randint(len(val_dataset))]

# Label Extraction task vector

## Train 

In [ ]:
prepend_bos = True

with_context = True
with_context = False

dtype = model.W_U.dtype
hist = []

# create Task Vector
TaskVec = torch.normal(mean=0, std=1.0, size=(1,dim), requires_grad=True, dtype=dtype)

# TaskVec = torch.ones((1,d), requires_grad=True)
for param in model.parameters():
    param.requires_grad = False

Now we train our Taskvector in order to retreive the labels. 
We use the cross entropy loss, *i.e* : $$\ell(x,y)=L=\{l_1,\ldots,l_N\}^\top,\quad l_n=-w_{y_n}\log\frac{\exp(x_{n,y_n})}{\sum_{c=1}^C\exp(x_{n,c})}\cdot1$$
The TransformerLens implementation is [here](https://github.com/TransformerLensOrg/TransformerLens/blob/f4287b68d544b5fe1221739a4f3c2424eccf7d31/transformer_lens/utils.py#L115), **Warning**, it takes as input all the logits, and perform the shift **afterwards** !

In [ ]:
#Training params
lr = 1e-2
batch_size = 20
epochs = 4
log_per_epoch = 2

if hist: 
    last_epoch = hist[-1]["epoch"]
else :
    last_epoch = 0
first_token_only = False
dtype = model.W_U.dtype
rep_idx = 1 if prepend_bos else 0 
taskVec_idx = rep_idx + 1
eos_tok = model.tokenizer.eos_token_id
eos_tok_str = model.tokenizer.eos_token

padding_tok = model.tokenizer.eos_token_id
replace_hook_name = tl.utils.get_act_name('embed') #pos_embed for gpt2 ... 

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
len_loader = len(train_dataloader)
n_log = len_loader // log_per_epoch
test_dataloader = DataLoader(test_dataset, batch_size=10, shuffle=True)
optim = torch.optim.Adam([TaskVec], lr=lr)
# optim = torch.optim.SGD([TaskVec], lr=lr)
# criterion = nn.CrossEntropyLoss()
#Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, len_loader//2, eta_min=lr/10)
# buffer to store best taskVec
best_TaskVec = torch.zeros_like(TaskVec)

gc.collect()
torch.cuda.empty_cache()
print(f"{'Resuming' if hist else 'Starting'} training TaskVec on {dataset_name} for layer {layer} of {model_name} with{'out' if not with_context else ''} context...")
for epoch in range(epochs):
    epoch += last_epoch
    b_count = 0
    m_loss = 0
    for batch in tqdm(train_dataloader):
        b_count += 1
                
        entities = batch["entity"]
        entity_toks = batch.get("entity_tokens", None)
        texts = batch["text"]
        reps = batch["representation"].squeeze(1).cuda()
        b_size = reps.shape[0]
        b_taskVec = TaskVec.repeat(b_size,1).cuda()
        
        if first_token_only:
            entities = [toks[0] for toks in model.to_str_tokens(entities, prepend_bos=False)]
        else:    
            entities = [ent + eos_tok_str for ent in entities] # take whole label add eos token

        if entity_toks is not None:
            eos = torch.tensor([eos_tok]).repeat(b_size,1)
            entity_toks = torch.cat([entity_toks, eos], dim=1).cuda()
        
        if with_context:
            prompts = [txt + "_ >" for txt in texts]
            context_toks = model.to_tokens(prompts, prepend_bos=prepend_bos, padding_side="left") 
            if entity_toks is None:
                entity_toks = model.to_tokens(entities, prepend_bos=False, padding_side="right")
            inputs = torch.cat([context_toks, entity_toks], dim=1)
            rep_idx = context_toks.shape[1] - 2
        else:
            rep_idx = 1 if prepend_bos else 0 
            
            if entity_toks is not None:
                prompts = ["_ >" for ent in entities]
                inputs = model.to_tokens(prompts, prepend_bos=prepend_bos,)
                inputs = torch.cat([inputs, entity_toks.cuda()], dim=1)
            else:
                prompts = ["_ > " + ent for ent in entities]
                inputs = model.to_tokens(prompts, prepend_bos=prepend_bos,)

        if b_count == 1:
            print(prompts, inputs)
            print(entities)
            print(entity_toks)
        rep_idxs = torch.tensor(b_size * [rep_idx])
        taskVec_idxs = torch.tensor(b_size * [rep_idx + 1])
        targets = inputs[:,rep_idx+1:] #don't take the '<eos>', <context>, '_' tokens into account

        logits = model.run_with_hooks(
                inputs,
                return_type = "logits",
                fwd_hooks=[
                    (replace_hook_name, utils.get_replace_with_rep_hook(reps, rep_idxs)), # replace '_' by the subject Representation
                    (replace_hook_name, utils.get_replace_with_rep_hook(b_taskVec, taskVec_idxs)) # replace 'called' by TaskVec Representation
                    ]
            ,)

        # LM Loss and optimization 
        loss = model.loss_fn(logits[:, rep_idx+1:,:], targets) # take only the loss on the entity tokens
        loss.backward()
        optim.step()
        optim.zero_grad()
        m_loss += loss.item()
        # scheduler.step()

        if b_count % n_log == 0:
            m_loss = m_loss / (len_loader // log_per_epoch)
            e = epoch + b_count/len_loader
            acc = eval_model(
                            model,
                            TaskVec,
                            test_loader=test_dataloader, 
                            first_token_only=first_token_only, 
                            with_context=with_context,
                            prepend_bos=prepend_bos)
            lr = scheduler.get_last_lr()[0]
            print(f"Epoch {e:.1f},  Batch {b_count}/{len_loader}, Loss: {m_loss:.3f}, Test Acc: {acc:.3f}, lr:{lr:.3f}")
            
            #save best taskVec according to test accuracy
            if hist and acc >= max([h["test accuracy"] for h in hist]):
                print(f" {acc:.3f} is the best accuracy so far, saving TaskVec ...")
                best_TaskVec[:] = TaskVec[:]

            hist.append({"epoch":e, "loss": m_loss, "test accuracy": acc, "lr":lr})
            m_loss = 0
            
task = "LabelExtractor" if first_token_only else "FirstTokenExtractor"
fileName = f'{task}_{model_name}_l{layer}_e{hist[-1]["epoch"]}.pth'


In [ ]:
tokens = [50256,    62,  1875, 15154, 50256]
str_tokens = model.tokenizer.convert_ids_to_tokens(tokens)
print('|'.join(str_tokens))

In [ ]:
item = train_dataset[np.random.randint(len(train_dataset))]
print(item)
str_token = " " + item["entity"]
tokens = model.to_tokens(item["text"]).view(-1)
print(tokens)
print(model.tokenizer.convert_ids_to_tokens(tokens))
model.get_token_position(str_token, tokens)

In [ ]:
import matplotlib.pyplot as plt

# Create a figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
epochs = [h["epoch"] for h in hist]
#plot lr in second axis for each 
ax2b = ax2.twinx()
ax2b.plot(epochs, [h["lr"] for h in hist], 'g--')
ax2b.set_ylabel('learning rate', color='g')
ax2b.tick_params(axis='y', labelcolor='g')
ax1b = ax1.twinx()
ax1b.plot(epochs, [h["lr"] for h in hist], 'g--')
ax1b.tick_params(axis='y', labelcolor='g')
# Plot the loss and accuracy
ax1.plot(epochs, [h["loss"] for h in hist])
ax2.plot(epochs, [h["test accuracy"] for h in hist])
# Set the labels and titles
ax1.set_xlabel("Epoch")
ax2.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_yscale('log') #log scale 
ax2.set_ylabel("Accuracy")
ax2b.set_ylabel("lr")
ax1.set_title("Loss")
ax2.set_title("Accuracy")
plt.show()

In [ ]:
task = "LabelExtractor" if not first_token_only else "FirstTokenExtractor"
task = "2tokensExtractor"
fileName = f'{task}_{model_name}_l{layer}_e{len(hist)//log_per_epoch}.pth'

print("Saving TaskVec to ", fileName)
torch.save(TaskVec, fileName)

## Testing the model

### Load TaskVec

In [ ]:

# load results
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
jobs_path = xp_path / "jobs" / xp_paths[0]
#get jobs
print("loading all results from ", jobs_path)
results = loadResults(jobs_path)


In [ ]:
from pathlib import Path
fileName = Path("./results/3320fdb2ffd15372a5a0b5a011871e86a696604b8335bfe874f2e363e1e91987/").glob("*.pth").__next__()
print("Loading TaskVec from ", fileName) 

In [ ]:
layer = 22
with_context = False
dataset_name = "TACRED"
dataset_name = "CoNLL2003"

method = "in_context"
method = "random_sample"

fileName = "./checkpoints/TaskVec_gpt2-small_l8_e50.pth"
fileName = "./LabelExtractor_gpt2-medium_l12_e9.pth"
fileName = "./checkpoints/TaskVec_mistral-7B_l17_e20.pth"
fileName = "./checkpoints/LabelExtractor_phi-2_l15_e20.pth"
fileName = "./checkpoints/contextLabelExtractor_phi-2_l15_e6.pth"
fileName = "./contextLabelExtractor_gpt2-medium_l15_e14.pth"
fileName = get_taskVec(results, 
                        model_name, 
                        layer=layer,
                        dataset_name=dataset_name,
                        extraction_method=method,
                        with_context = with_context)


In [ ]:


# Get TaskVec from the model
TaskVec = torch.load(fileName)
print("TaskVec loaded from ",
 fileName)

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=20, shuffle=True)
eval_model(model, TaskVec, test_dataloader, prepend_bos=True, with_context=True)

### Compute metrics

In [ ]:
with_context = False
# layer=15
data = test_dataset
# data = test_dataset.data[:100]
# test_dataset.data = data
# test_dataset.augment_with_repr(model, layer, batch_size=100)

# for d in data:                  # drop inferered key in data
#     d.pop("inferred", None)
acc = evaluateTV(model, TaskVec,
                       data,
                       with_context=with_context, 
                       max_tokens=10, b_size=5)
print(f" Model {model_name}, layer {layer}:\n Evaluation metrics on test set:")
for k,v in acc.items():
    print(f"\t- {k}: {v:.3f}")

In [ ]:
evaluateTV(model, best_TaskVec,
                       test_dataset,
                       with_context=with_context, 
                       max_tokens=20, b_size=30, prepend_bos=True)

In [ ]:
#print some fail examples
print("Original:".center(40) + " | " + "Inferred:".center(40)+ '\n'+"-"*84)
limit = 10
fails = set()
for item in data:
    if len(fails) >= limit: break
    item = data[np.random.randint(len(data))]
    if item['entity'] == item['inferred'] or item['inferred'] in fails: continue
    fails |= {item['inferred']}
    print(f"{item['entity'].center(40)} | {item['inferred'].center(40)} | {item['text']}")

In [ ]:
#print some with context
print("Original:".center(40) + " | " + "Inferred:".center(40)+ '\n'+"-"*84)
limit = 10
fails = set()
for item in data:
    if len(fails) >= limit: break
    item = data[np.random.randint(len(data))]
    if item['entity'] == item['inferred'] or item['inferred'] in fails: continue
    fails |= {item['inferred']}
    print(f"\n{item['entity'].center(40)} | {item['inferred'].center(40)}")
    print(f"context: {item['text']}")

In [ ]:
dataset = test_dataset
ind = np.random.randint(len(dataset))
prompt = dataset[ind]["text"]
print(prompt)
print("target:", dataset[ind]["entity"])
# s = dataset[ind]["representation"].view(1,-1)

# s = repr
# gen = utils.generate_from_repr(model, s, TaskVec, retr_prompt="_ >", max_tokens=10, do_sample=True)
# print("generation:","".join(gen[3:-1]))

In [ ]:

def generate_from_repr( model, 
                        repr, 
                        taskVector = None, 
                        retr_prompt:str ="_ named",
                        max_tokens = 10,
                        do_sample = False,
                        prepend_bos=True,
                        ):
        """
        Args:
                repr: The representation vectors
                Model: the GPT model to use
                taskVector: the task vector to use. If None, will use default prompt
                max_tokens: num of tokens to generate
                do_sample: if True, sample from logits, else take argmax
                prepend_bos: if True, prepend '<BOS>' token to the input

        """
        inp_toks = model.to_tokens(retr_prompt, prepend_bos=prepend_bos)
        rep_idx = 1 if prepend_bos else 0
        replace_hook_name = tl.utils.get_act_name('embed')
        taskVec_idx = rep_idx + 1
        
        if taskVector is not None:
            taskVector = taskVector.view(1,-1)
            b_taskVec = taskVector.repeat(repr.shape[0],1).cuda()
        
        repr = repr.cuda()
        # print(b_taskVec.shape)
        # print(repr.shape)
        
        for i in range(max_tokens):
                # print(inputs, targets)
                if taskVector is not None:
                    logits = model.run_with_hooks(
                            inp_toks,
                            return_type = "logits",
                            fwd_hooks=[
                            (replace_hook_name, utils.get_replace_with_rep_hook(repr, torch.tensor([rep_idx]))), # replace '_' by the subject Representation
                            (replace_hook_name, utils.get_replace_with_rep_hook(b_taskVec, torch.tensor([taskVec_idx]))) # replace 'called' by TaskVec Representation
                            ])
                else:
                    logits = model.run_with_hooks(
                            inp_toks,
                            return_type = "logits",
                            fwd_hooks=[
                            (replace_hook_name, utils.get_replace_with_rep_hook(repr, torch.tensor([rep_idx]))), # replace '_' by the subject Representation
                            ])

                final_logits =  logits[0,-1,:] #extract logits for last token 

                if do_sample:
                        new_tok = tl.utils.sample_logits(
                        final_logits,
                        top_k=None,
                        top_p=None,
                        temperature=0,
                        freq_penalty=0,
                        tokens=inp_toks,
                        ).view(1,-1)
                else:       #greedy generation
                        new_tok = final_logits.argmax(-1).view(1,-1) 
                
                inp_toks = torch.hstack((inp_toks,new_tok))
                # stop if EOS token
                if new_tok == model.tokenizer.eos_token_id: break
        print(inp_toks)
        return model.tokenizer.decode(inp_toks.view(-1)[1:])

In [ ]:
limit = 1000
layer = 17
test_dataset.data = test_dataset.data[:limit]
print("length:",len(test_dataset))
test_dataset.augment_with_repr(model, layer, batch_size=20)

In [ ]:
## Check how does the model reads the TaskVec
utils.generate_from_repr(model, repr, TaskVec, retr_prompt="_ >", do_sample=True)

## Conclusion
The model is able to generate text from a representation of a subject.
It can do it for known entities, but likely not for representations of an unknown label (unknow name), even if there are clues that it knows where to look.
- adding the representation makes Phi1.5 model jump from 75 to 85% LM accuracy.
- But only 25% for the first token, whereas it can reach 32% if it is trained solely on predicting the first token 


# ICL Baseline
Following the works from [Patchscopes](https://openreview.net/forum?id=5uwBzcn885), we can try to use in-context learning to infer the entity labels.

In [ ]:
dataset = train_dataset
icl_prompt = ';'.join([f"{ent['entity']}>{ent['entity']}" for ent in dataset[:3]])
print(icl_prompt)
prompt = f"{icl_prompt};_ >"
print('|'.join(model.to_str_tokens(prompt)))

In [ ]:
from LabelExtractor import infer_entities_icl, compute_metrics

# data = val_dataset 
data = test_dataset 

inferred_data = infer_entities_icl(model, data, b_size=5, n_examples=5)

print(compute_metrics(inferred_data))

In [ ]:

for i in range(10):
    ind = np.random.randint(len(inferred_data))
    print(f"Context: '{inferred_data[ind]['text']}'\nOriginal: {inferred_data[ind]['entity']}\nInferred: {inferred_data[ind]['inferred']}\n{'-'*80}")

# Convergence of representations
During Representation Construction
Here, we explore how the representation that we extract from the model is constructed by the Transformer.

### Compute cache

In [ ]:
in_voc_space = False

#get one example from the dataset
ind = np.random.randint(len(test_dataset))
# ind = 9296
# ind = 64
ind = 2574
print(f"sample {ind} from test dataset")
target = test_dataset[ind]["entity"]
prompt = test_dataset[ind]["text"].split(target)[0] + target
toks = model.to_str_tokens(prompt)

#tokenize the prompt
print( "prompt:", test_dataset[ind]["text"])
print( "|".join(toks))
print(f"{len(toks)} tokens, target:{target}")

layer = 20
hooks = [ # name, type, color
        # ["normalized","ln1", "green"],
        ["attn_out","", "blue"],
        # ["normalized","ln2", "lightgreen"],
        ["mlp_out","", "cyan"],
        ["resid_post","","red"]
        ]
# print hooks 
print("will use hooks:")          
for h,t,_ in hooks:
    hook = tl.utils.get_act_name(h, 0, t)
    print(hook)
similarities = torch.zeros(len(toks), len(hooks) * model.cfg.n_layers)
random_similarities = torch.zeros(len(hooks) * model.cfg.n_layers)
n = len(hooks)

#mean similarity between vectors in a batch of representations
@torch.no_grad()
def mean_similarities(reprs):
    sims = []
    for i in range(reprs.shape[0]-1):
        sim = torch.cosine_similarity(reprs[i], reprs[i+1:], dim=-1)
        sims.append(sim.mean().cpu().detach())
    return torch.tensor(sims).mean().cpu().detach()

#get whole hidden states
_ , cache = model.run_with_cache(prompt)
repr = cache[tl.utils.get_act_name("resid_post", layer, "")][:,-1,:] # 1 x n_tokens x dim
repr = repr.cuda().view(1,-1) 
if in_voc_space:
    repr = repr @ model.W_U #transform in vocab space
for l in range(model.cfg.n_layers):
    for i, (h,t,_) in enumerate(hooks):
        hook = tl.utils.get_act_name(h, l, t)
        reprs = cache[hook] # 1 x n_tokens x dim : representations of the tokens at given hook
        reprs = reprs.type(model.W_U.dtype) #cast to dtype of model
        if in_voc_space:
            reprs = reprs @ model.W_U
        #compute similarities with the target representation
        sim = torch.cosine_similarity(
            reprs,
            repr,   
            dim=-1)
        similarities[:,n*l+i] = sim.squeeze(0)
        random_similarities[n*l+i] = mean_similarities(reprs.view(-1,reprs.shape[-1]))

#add embeddings
hook = tl.utils.get_act_name("embed")
reprs = cache[hook].squeeze(0) # n_tokens x dim
sim = torch.cosine_similarity(reprs,
                              repr,     # already in vocab space
                              dim=-1).cpu().detach().view(-1,1)
similarities = torch.hstack((sim, similarities )).cpu().detach().numpy()

del cache
gc.collect()
torch.cuda.empty_cache()

## Plotting Convergence to extracted representation

#### Plot Everything

In [ ]:
import matplotlib.pyplot as plt

#show image of similarities
plt.figure(figsize=(16,16))
plt.imshow(similarities.T, aspect='auto', cmap='bwr')
plt.colorbar()
#colorbar with scale from 0 to 1
#draw lines separating layers
for i in range(model.cfg.n_layers + 1):
    plt.axhline(y=n*i+.5, color='black', linewidth=0.5)
plt.clim(-1,1)
plt.title("Similarities between tokens and target representation")
plt.ylabel(f"Layers in {model_name}")
plt.gca().invert_yaxis() #invert y axis
plt.yticks(np.array(range(0,1 + n*model.cfg.n_layers,n)), ["Embed"] + [f"Layer {i}" for i in range(model.cfg.n_layers)])
plt.xlabel("Tokens")
plt.xticks(range(len(toks)), toks, rotation=90)
plt.show()

#### Plot output of all layers

In [ ]:
outputs_only = np.zeros((len(toks), model.cfg.n_layers))
for i in range(model.cfg.n_layers):
    outputs_only[:,i] = similarities[:,n*(i+1)] # 1 + n*(i+1) - 1
#show image of similarities
plt.figure(figsize=(10,10))
plt.imshow(outputs_only.T, aspect='auto', cmap=plt.cm.YlGnBu) #'bwr')
plt.colorbar()
#colorbar with scale from 0 to 1
#draw lines separating layers
for i in range(model.cfg.n_layers):
    plt.axhline(y=i+.5, color='black', linewidth=0.5)
plt.clim(-1,1)
plt.title("Similarities between tokens and target representation")
plt.ylabel(f"Layers in {model_name}")
plt.gca().invert_yaxis() #invert y axis
plt.yticks(np.array(range(0,model.cfg.n_layers)), [f"{i}" for i in range(model.cfg.n_layers)])
plt.xlabel("Tokens")
plt.xticks(range(len(toks)), toks, rotation=90)
plt.show()

#### Plot for Last token

In [ ]:
# Plot the last token similarities
plt.figure(figsize=(13, 4))

unique_colors = ["black"] + [c for _, _, c in hooks]
legend_labels = ["Embed"] + [h for h, _, _ in hooks]  # Labels corresponding to the unique colors
colors = ["black"] + [c for _, _, c in hooks] * model.cfg.n_layers
x = np.array(range(1, len(random_similarities)+1))
# plt.grid(axis='y')
plt.bar(range(similarities.shape[1]), similarities[-1, :], color=colors)
plt.hlines(random_similarities, x-.5, x+.5, color='black', linewidth=0.5, label="Random")
# plt.step([0.5] + list(x+.5), [0] + list(random_similarities.numpy()), color='black', linewidth=0.5, label="Random")
plt.title(f"Similarities of last token representations in {model_name}")
plt.ylabel("Cosine similarity")
plt.xlabel("Layers")
plt.xticks(range(0, n * model.cfg.n_layers + 1, n), ["emb"] + [f"{i+1}" for i in range(model.cfg.n_layers)])
plt.xlim(-4, similarities.shape[1] + 25)
# Create a custom legend with one entry per unique color
# Create a list of unique colors and labels

# Plot the legend manually by creating proxy artists (handles) for each color
handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in unique_colors]
# plt.legend()
plt.legend(handles, legend_labels, 
           title="SubLayers",
            # loc="lower left",
            )
# Display the plot
plt.savefig("similarities.pdf")
plt.show()

#### Plot for given Layer

In [ ]:
#plot the last token similarities
plt.figure(figsize=(16,5))

outputs = [similarities[i,n*(layer+1)] for i in range(len(toks))]
plt.bar(range(len(outputs)), outputs)

plt.title(f"Similarities of token representations at layer {layer} in {model_name}")
plt.ylabel("Cosine similarity")
plt.xlabel("Tokens")
plt.xticks(range(len(toks)), toks, rotation=90)
plt.legend()
plt.show()

# SubLayer Knockout

In [ ]:
#get one example from the dataset
ind = np.random.randint(len(test_dataset))
# ind = 9296
print(f"sample {ind} from test dataset")
target = test_dataset[ind]["entity"]
prompt = test_dataset[ind]["text"].split(target)[0]+ target # 'in_context' method
toks = model.to_str_tokens(prompt)

#tokenize the prompt
print( "prompt:", test_dataset[ind]["text"])
print( "|".join(toks))
print(f"{len(toks)} tokens, target:{target}")

In [ ]:
layer = 17 # layer where to extract representations
to_voc_space = True
to_voc_space = False

hooks = [ # name, type, color
        ["attn_out","", "blue"], 
        ["mlp_out","", "cyan"],
        ]

#tokenize the prompt
tokens = model.to_tokens(prompt)
rep_ind = tokens.shape[-1] - 1
#get representations
repr = utils.get_representation(model, tokens=tokens, token_inds=[rep_ind], layer=layer, verbose=True)
# cast to model dtype
print(repr.shape, repr.dtype)
#transform in vocab space
if to_voc_space: repr = repr.cuda().view(1,-1) @ model.W_U


In [ ]:
# Compute similarities with the target representation and its knocked versions
print("will knock out hooks:")          
for h,t,_ in hooks:
    hook = tl.utils.get_act_name(h, 0, t)
    print(hook)

def get_knock_hook(ind):
    def Knockout_hook(tensor, ind, hook):
        tensor[:,ind,:] = 0 # batch, seq len, dim
        return tensor
    return lambda tensor , hook: Knockout_hook(tensor, ind, hook)

similarities = torch.zeros(tokens.shape[-1], len(hooks) * (layer + 2))
n = len(hooks)
for l in tqdm(range(layer+2)):
    for i, (h,t,_) in enumerate(hooks):
        for ind in range(tokens.shape[-1]):
            hook = tl.utils.get_act_name(h, l, t)
            knocked_repr = utils.get_representation(
                    model, 
                    tokens=tokens, 
                    token_inds=[rep_ind], 
                    layer=layer,
                    hooks = [(hook, get_knock_hook(ind))],
                    # verbose=True,
                    )
            knocked_repr = knocked_repr.type(model.W_U.dtype) #cast to dtype of model
            if to_voc_space: knocked_repr = knocked_repr.cuda().view(1,-1) @ model.W_U
            #compute similarities with the target representation
            sim = torch.cosine_similarity(
                # knocked_repr @ model.W_U,    # transform in vocab space
                knocked_repr,
                repr, 
                dim=-1)
            similarities[ind,n*l+i] = sim.squeeze(0).cpu().detach()

gc.collect()
torch.cuda.empty_cache()

In [ ]:
#show image of similarities
plt.figure(figsize=(16,16))
#reverse colorbar
plt.imshow(similarities.T, aspect='auto', cmap=plt.cm.YlGnBu) #'bwr')
plt.colorbar()
#colorbar with scale from 0 to 1
#draw lines separating layers
for i in range(layer + 1):
    plt.axhline(y=n*i-.5, color='black', linewidth=0.5)
plt.clim(0,1)
plt.title("Similarities between tokens and target representation")
plt.ylabel(f"Layers in {model_name}")
plt.gca().invert_yaxis() #invert y axis
plt.yticks(np.array(range(0,n*layer,n)), [f"Layer {i}" for i in range(layer)])
plt.xlabel("Tokens")
plt.xticks(range(len(toks)), toks, rotation=90)
plt.show()

In [ ]:
#plot the last token similarities
plt.figure(figsize=(16,5))

#color all multiples of n in red 
colors = [c for _,_,c in hooks]*(model.cfg.n_layers)
labels = [h for h,_,_ in hooks]*(model.cfg.n_layers)

plt.bar(range(similarities.shape[-1]),similarities[-1,:],
        #  label=labels, 
         color=colors)

plt.title(f"Similarities of last token representations in {model_name} \n with sublayer knockout")
plt.ylabel("Cosine similarity")
plt.xlabel("Layers")
plt.xticks(range(0, n*model.cfg.n_layers, n), [f"l{i}" for i in range(model.cfg.n_layers)])
plt.legend(labels, title="Sublayers", loc='upper left')
plt.show()

In [ ]:
# quick sanity check 
_ , cache = model.run_with_cache(toks)      #get whole hidden states
repr2 = cache[tl.utils.get_act_name("resid_post", layer, "")][:,-1,:] # 1 x n_tokens x dim
repr2.cpu().allclose(repr)


# Attention Dissection

In [ ]:
import circuitsvis as cv
import plotly.io as pio
# Plotly needs a different renderer for VSCode/Notebooks vs Colab argh
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")
# Testing that the library works
cv.examples.hello("Victor")


In [ ]:
#get one example from the dataset
ind = np.random.randint(len(test_dataset))
# ind = 7989 # 'New Hampshire' - can be regenerated without context at layer 15
# ind = 4237 # 'English-language' - can almost be regenerated without context at layer 15
target = test_dataset[ind]["entity"]
context = test_dataset[ind]["text"]
print(f"sample {ind} from test dataset. Target: {target}")
prompt = context + ' ' + target # prompt used to build the representations
str_tokens = model.to_str_tokens(prompt)

#get whole cache for given prompt as it is done in 'augment_with_repr'
_ , cache = model.run_with_cache(prompt)

#display the prompt
html = cv.tokens.colored_tokens(str_tokens,[])
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 50px")
html
#add padding at the end of html

## Attention while constructing the representation
Here, we just explore the use of `cv.attention.attention_patterns` function from `circuitsvis` that allows us to inspect the attention scores for each head of our model when run on the prompt that was used to retreive the representations

In [ ]:
layer = 10
print(type(cache))
attention_pattern = cache["pattern", layer, "attn"].squeeze(0)
print("pattern shape:", attention_pattern.shape)
print(f"Layer {layer} Head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

## Attention while generating labels
Now, we explore what is happening on the attention side when we perform the label generation. 

### Retrieve corresponding Trained TaskVector

In [ ]:
layer = 15
with_context = True
dataset_name = "TACRED"
# load results
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
jobs_path = xp_path / "jobs" / xp_paths[0]
#get jobs
print("loading all results from ", jobs_path)
results = loadResults(jobs_path)
# Get TaskVec from the model
fileName = get_taskVec(results, model_name, layer=layer, dataset_name=dataset_name, with_context = with_context)
TaskVec = torch.load(fileName)
print("TaskVec loaded from ", fileName)

### Generate with TaskVec

In [ ]:
#get all representations at given layer from cache
with_context = False
with_context = True
layer = 15

repr = cache[tl.utils.get_act_name("resid_post", layer, "")][0,-1,:] # n_tokens x dim
repr = repr.detach().cuda()
data = [{"id": 0, "text": context, "representation": repr, }]
infer_entities(model, TaskVec, data, with_context=with_context,
                return_attn_pattern=True,
                )
print("inferred '",data[0]["inferred"],"' from representation extracted at layer", layer, "of model", model_name) 


In [ ]:
attn_layer = 10
#get the attention pattern for the inferred entity at the given layer
pattern = data[0]["attn_pattern"][attn_layer]
str_tokens = model.to_str_tokens((context + ' _ > ' if with_context else '_ > ') + data[0]["inferred"])
print(f"Layer {attn_layer} Head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=pattern)

In [ ]:
#use 'attention_pattern' to display only for one head
cv.attention.attention_heads(tokens=str_tokens, attention=pattern)

# Train Linear Layer to filter representations

### Load TaskVec

In [ ]:
# load results
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
jobs_path = xp_path / "jobs" / xp_paths[0]
#get jobs
print("loading all results from ", jobs_path)
results = loadResults(jobs_path)

In [ ]:
with_context = True
with_context = False

train_TaskVec = False
train_TaskVec = True

# Get TaskVec from the model
fileName = get_taskVec(results, model_name, layer=layer, dataset_name=dataset_name, with_context = with_context)
dtype = model.W_U.dtype
TaskVec = torch.load(fileName).type(dtype).detach().cuda()
print("TaskVec loaded from ", fileName)

if train_TaskVec:
    TaskVec.requires_grad_(True)

### Test it

In [ ]:
# with_context = False #test generalization without context
# with_context = True
print(f"Computing metrics on {dataset_name} dataset with {model_name} model at layer {layer} {'with' if with_context else 'without'} context...")
evaluateTV(model, TaskVec, test_dataset, with_context=with_context, max_tokens=100, b_size=5, prepend_bos=True)

In [ ]:
#pop inferred key in data
for d in test_dataset:
    d.pop("inferred", None)

## Train Projection

In [ ]:
dim = model.QK.shape[-1]
linear_model = torch.nn.Linear(dim, dim, bias=True)
#initialize the linear model with identity
linear_model.weight.data = torch.eye(dim).type(dtype)
linear_model.bias.data = torch.zeros(dim, dtype=dtype)
linear_model = linear_model.cuda()

hist = []
#freeze the model and the task vector
for param in model.parameters():
    param.requires_grad = False

In [ ]:
#Training params
lr = 1e-3
batch_size = 30
epochs = 5
logs_per_epoch = 4
clip = 1.0 #gradient clipping
prepend_bos = True
first_token_only = False

if hist: 
    last_epoch = hist[-1]["epoch"]
else :
    last_epoch = 0
dtype = model.W_U.dtype
eos_tok_str = model.tokenizer.eos_token
replace_hook_name = tl.utils.get_act_name('embed') #pos_embed for gpt2 ... 

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
n_log = len(train_dataloader) // logs_per_epoch
val_dataloader = DataLoader(val_dataset, batch_size=10, shuffle=True)

# optim = torch.optim.Adam([TaskVec] + list(linear_model.parameters()) , lr=lr)
optim = torch.optim.Adam(linear_model.parameters() , lr=lr)

#Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, len_loader//2, eta_min=lr/10)

gc.collect()
torch.cuda.empty_cache()
print(f"{'Resuming' if hist else 'Starting'} training TaskVec on {dataset_name} for layer {layer} of {model_name} with{'out' if not with_context else ''} context...")
b_count = -1
for epoch in range(epochs):
    epoch += last_epoch
    m_loss = 0
    for batch in tqdm(train_dataloader):
                
        b_count += 1
        entities = batch["entity"]
        texts = batch["text"]
        reps = batch["representation"].squeeze(1).cuda()
        b_size = reps.shape[0]
        b_taskVec = TaskVec.repeat(b_size,1).cuda()
        
        if first_token_only:
            entities = [toks[0] for toks in model.to_str_tokens(entities,prepend_bos=False)]
        else:    
            entities = [ent + eos_tok_str for ent in entities] # take whole label add eos token

        if with_context:
                    prompts = [txt + "_ >" for txt in texts]
                    context_toks = model.to_tokens(prompts, prepend_bos=prepend_bos, padding_side="left") 
                    entities_toks = model.to_tokens(entities, prepend_bos=False, padding_side="right")
                    inputs = torch.cat([context_toks, entities_toks], dim=1)
                    rep_idx = context_toks.shape[1] - 2
        else:
            rep_idx = 1 if prepend_bos else 0 
            prompts = ["_ > " + ent for ent in entities]
            inputs = model.to_tokens(prompts, prepend_bos=prepend_bos,)

        rep_idxs = torch.tensor(b_size * [rep_idx])
        taskVec_idxs = torch.tensor(b_size * [rep_idx + 1])
        targets = inputs[:,rep_idx+1:] #don't take the '<eos>', <context>, '_' tokens into account

        #transform the representations with linear model
        reps = linear_model(reps)

        logits = model.run_with_hooks(
                inputs,
                return_type = "logits",
                fwd_hooks=[
                    (replace_hook_name, utils.get_replace_with_rep_hook(reps, rep_idxs)), # replace '_' by the subject Representation
                    (replace_hook_name, utils.get_replace_with_rep_hook(b_taskVec, taskVec_idxs)) # replace 'called' by TaskVec Representation
                    ]
            ,)

        # LM Loss and optimization 
        loss = model.loss_fn(logits[:, rep_idx+1:,:], targets) # take only the loss on the entity tokens
        loss.backward()
        optim.step()
        optim.zero_grad()
        m_loss += loss.item()
        
        # scheduler.step()
        
        if b_count % n_log == 0:
            m_loss = m_loss / n_log
            e = epoch + b_count/len_loader
            acc = eval_model(
                            model,
                            TaskVec,
                            val_loader=val_dataloader, 
                            first_token_only=first_token_only, 
                            with_context=with_context,
                            prepend_bos=prepend_bos,
                            transform=linear_model,
                            # metric='loss'
                            )
            lr = scheduler.get_last_lr()[0]
            print(f"Epoch {e:.1f},  Batch {b_count}/{len_loader}, Loss: {m_loss:.3f}, Test Acc: {acc:3f}, lr:{lr:.4f}")
            
            #save best taskVec according to test accuracy
            # if hist and acc >= max([h["test accuracy"] for h in hist]):
            #     print(f" {acc:.3f} is the best accuracy so far, saving TaskVec ...")
            #     best_TaskVec[:] = TaskVec[:]
            hist.append({"epoch":e, "loss": m_loss, "test accuracy": acc, "lr":lr})
            m_loss = 0
    b_count = 0        

task = "LabelExtractor" if first_token_only else "FirstTokenExtractor"
fileName = f'{task}_{model_name}_l{layer}_e{hist[-1]["epoch"]}.pth'


In [ ]:
#save linear model
torch.save(linear_model, f"./checkpoints/LinearModel_CoTrained_{model_name}_l{layer}_{'' if with_context else 'no'}Context.pth")

In [ ]:
# plot the training history
import matplotlib.pyplot as plt

# Create a figure and two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

epochs = [h["epoch"] for h in hist]
losses = [h["loss"] for h in hist]
test_acc = [h["test accuracy"] for h in hist]
# Plot the loss and accuracy
ax1.plot(epochs, losses)
ax2.plot(epochs, test_acc)
#log 
ax1.set_yscale('log')
# Set the labels and titles
ax1.set_xlabel("Epoch")
ax2.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
plt.show()


## Analysis

In [ ]:
### Plot learned weights of the linear model
W = linear_model.weight.float().cpu().detach().numpy()
print(f"Weights of the linear model: {W.shape}, max:{W.max():.2f}, min:{W.min():.2f}")
print(f"Biases of the linear model: {linear_model.bias.shape}, max:{linear_model.bias.max():.2f}, min:{linear_model.bias.min():.2f}")
#eigenvalues of the weight matrix
eigvals = np.linalg.eigvals(W)
print(f"Eigenvalues of the weight matrix: max:{eigvals.max():.2f}, min:{eigvals.min():.2f}")
#plot the weights of the linear model
plt.figure(figsize=(16,5))
plt.imshow(W)
plt.colorbar()
plt.title("Linear model weights")


In [ ]:
#svd for the learned weights
with torch.no_grad():
    U, S, Vh = torch.linalg.svd(
        linear_model.weight.float(),
        full_matrices=True)
N = 40
# plot of the singular values and a zoom on the first N
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].plot(S.detach().cpu().numpy())
ax[0].set_title("Singular values of the learned weights")
ax[1].plot(S.detach().cpu().numpy()[:N], 'o-')
ax[1].set_title(f"Singular values of the learned weights (first {N})")
ax[1].grid()
ax[2].plot(S.detach().cpu().numpy()[-N:], 'o-')
ax[2].set_title(f"Singular values of the learned weights (last {N})")
ax[2].grid()
plt.show()

In [ ]:
import copy
# test_dataset = augmented_data
#build new dataset with cleaned representations:

cleaned_test_dataset = copy.deepcopy(test_dataset)

#get repr
cleaned_test_dataset.augment_with_repr(model, layer, batch_size=10)

#clean representations
for item in tqdm(cleaned_test_dataset):
    item["representation"] = linear_model(item["representation"].cuda()).cpu().detach()

In [ ]:
i = np.random.randint(len(cleaned_test_dataset))
#check difference between raw and cleaned representations
print(cleaned_test_dataset[i]["representation"])
print(test_dataset[i]["representation"])
print(f"norm of raw representation: {torch.norm(test_dataset[i]['representation']):.3f}")
print(f"norm of cleaned representation: {torch.norm(cleaned_test_dataset[i]['representation']):.3f}")
print(f"similarity: {torch.cosine_similarity(cleaned_test_dataset[i]['representation'], test_dataset[i]['representation'], dim=-1):.3f}")

In [ ]:
print(f"Computing score with cleaned? representations: {model_name} model at layer {layer} {'with' if with_context else 'without'} context...")
print(evaluateTV(model=model, 
                      TaskVec=TaskVec, 
                      test_dataset=cleaned_test_dataset,
                      with_context=with_context,
                      b_size=5,
                      force_recompute=True,
                      ))

In [ ]:
for i in range(len(test_dataset)):
    # print(d["text"])
    d = test_dataset[i]
    d_c = cleaned_test_dataset[i]
    success = d["entity"].strip() == d["inferred"].strip()
    if not success:
        print(d['text'])
        print(f"target: '{d['entity'].strip()}' | inferred:'{d['inferred']}' | inf_cleaned: '{d_c['inferred']}'\n")

# Tests with linear Layers

## Load Linear models

In [ ]:
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
#load linear layers training results
xp_lin = 'labelextractor.learnlinearfilter'

#get jobs
results_linear = loadResults(xp_path / "jobs" / xp_lin)
print(f"{len(results_linear)} jobs loaded")
#remove jobs with no Eval
# results_linear = results_linear[results_linear["Eval"].notna()]
results_linear["hash"] = results_linear["path"].apply(lambda x: x.parts[-1])
print(f"{len(results_linear)} jobs with eval")
# results_linear.head()

In [ ]:
layer = 10
with_context = False
dataset_name = "CoNLL2003"
method = "in_context"
method = "average"

filtered_results = results_linear[
    (results_linear["layer"] == layer) & 
    (results_linear["dataset_name"] == dataset_name) & 
    (results_linear["with_context"] == with_context) &
    (results_linear["extraction_method"] == method)
    ]
filtered_results.head()

## Analysing linear models from same layer

In [ ]:
linear_models = []
for i, row in filtered_results.iterrows():
    path = row["path"]
    #open folder
    files = os.listdir(path)
    ckpts = [f for f in files if f.endswith(".pth")]
    if ckpts:
        #load last checkpoint
        ckpt = torch.load(path / ckpts[-1])
        linear_models.append(ckpt)
        print(f"loaded {ckpts[-1]} from {path}")
    else:
        print(f"Job {row['hash']} has {len(ckpts)} checkpoints")


In [ ]:
N = 10
first_components = []
last_components = []

#svd for the learned weights
for linear_model in linear_models:
    with torch.no_grad():
        U, S, Vh = torch.linalg.svd(
            linear_model.weight.float(),
            full_matrices=True)
    first_components.append(Vh[:N,:].cpu().numpy())
    last_components.append(Vh[-N:,:].cpu().numpy())
    # plot of the singular values and a zoom on the first N
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    ax[0].plot(S.detach().cpu().numpy())
    ax[0].set_title("Singular values of the learned weights")
    ax[1].plot(S.detach().cpu().numpy()[:N], 'o-')
    ax[1].set_title(f"Singular values of weights (first {N})")
    ax[1].grid()
    ax[2].plot(S.detach().cpu().numpy()[-N:], 'o-')
    ax[2].set_title(f"Singular values of weights (last {N})")
    ax[2].grid()
    plt.show()

In [ ]:

def plot_sims(components, i_comp):
    plt.figure(figsize=(12, 6))
    similarities = np.zeros((len(components), len(components)))

    for i, c1s in enumerate(components):
        for j, c2s in enumerate(components):
            c1 = c1s[i_comp,:]
            c2 = c2s[i_comp,:]
            # print(c1.shape, c2.shape)
            #compute similarity
            if i <= j:
                similarities[i,j] = np.abs(np.dot(c1,c2)) / (np.linalg.norm(c1) * np.linalg.norm(c2))
            else:
                similarities[i,j] = np.nan

    plt.title(f"Similarity between {i_comp}th singular vectors of linear models")
    plt.imshow(similarities, cmap=plt.cm.YlGnBu)
    plt.colorbar()
    plt.clim(0,1)
    plt.show()

plot_sims(first_components, 0)
plot_sims(last_components, -1)

In [ ]:
from scipy.linalg import subspace_angles

def plot_angles(components1, components2, dist="grassmann"):
    """
    components: list of numpy arrays of shape (n_components, n_features)
    dist: distance metric to use, either 'grassmann' or 'max' 
        'grassmann' computes the grassmann distance between the subspaces spanned by the components
        'max' computes the maximum principal angle between the subspaces spanned by the components, **in degrees**
    """
    angles = np.zeros((len(components1), len(components2)))

    for i, c1s in enumerate(components1):
        for j, c2s in enumerate(components2):
            
            #compute similarity
            if i <= j:
                all_angles = subspace_angles(c1s.T, c2s.T) #suppose c1s and c2s are (n_features, n_components) : vectors are columns
                if dist == "grassmann":
                    angles[i,j] = np.sqrt(np.sum(all_angles**2))
                elif dist == "mean":
                    angles[i,j] = np.rad2deg(np.mean(all_angles))
                elif dist == "max":
                    angles[i,j] = np.rad2deg(np.max(all_angles))
                else: 
                    raise ValueError(f"Unknown distance metric: {dist}")
            else:
                angles[i,j] = np.nan

    #print mean angle is scientific notation
    print(f"mean {dist}: {np.nanmean(angles):.2f}{'' if dist == 'grassmann' else '°'}")
    plt.title(f"max principal subspaces angle \n of linear models at layer {layer}")
    plt.imshow(angles, cmap=plt.cm.YlGnBu)
    plt.colorbar()
    if dist == 'max': 
        plt.clim(0, 90)
    plt.show()

#baseline, compute angle between first and last components
print(f"using the first {N} components")
print(f"baseline, angles between random {N}-dimensional spaces")
shape = first_components[0].shape
comp1 = np.random.rand(shape[1],shape[0])
comp2 = np.random.rand(shape[1],shape[0])
angles = subspace_angles(comp1, comp2)
print(f"mean angle between two random subspaces: {np.rad2deg(np.mean(angles)):.2f}°")
print(f"grassmann between two random subspaces: {np.sqrt(np.sum(angles**2)):.2f}\n")

print(f"angles between first component spaces of {len(linear_models)}linear models")
dist = "max"
dist = "mean"

plot_angles(first_components, first_components, dist=dist)

print(f"angles between last component spaces of {len(linear_models)} linear models")
plot_angles(last_components, last_components, dist=dist)


print(f"angles between first and last component spaces of {len(linear_models)} linear models")
plot_angles(first_components, last_components, dist=dist)

In [ ]:
## project first components on vocab
#get tok 10 mappings:
def project_on_vocab(model, rep, k=10):
    """ project a representation on the vocabulary of the model
    Args:
        model: the model to use
        rep: the representation to project
        k: number of top tokens to return
    """
    #get top 10 tokens
    logits = torch.tensor(rep).float().cuda() @ model.W_U
    topk_inds = torch.topk(logits, k).indices
    topk_tokens = model.tokenizer.convert_ids_to_tokens(topk_inds.cpu().numpy())
    return topk_tokens

print("first components projections on vocab")
print(50*"-")
for comps in first_components:
    comp = comps[0]
    print(project_on_vocab(model, comp, k=10))

print("\nlast components projections on vocab")
print(50*"-")
for comps in last_components:
    comp = comps[0]
    print(project_on_vocab(model, comp, k=10))

### Similariy of biases

In [ ]:
#compute the similarity between linear models biases 
biases = [m.bias.detach().cpu().numpy() for m in linear_models]

#plot the biases norm 
plt.figure(figsize=(5, 3))
norms = [np.linalg.norm(b) for b in biases]
plt.scatter(range(len(norms)), norms)
plt.title("Norm of biases of linear models")
plt.show()

plt.figure(figsize=(12, 6))
similarities = np.zeros((len(biases), len(biases)))

for i, b1 in enumerate(biases):
    for j, b2 in enumerate(biases):
        if i <= j:
            similarities[i,j] = np.abs(np.dot(b1,b2)) / (np.linalg.norm(b1) * np.linalg.norm(b2))

plt.title(f"Similarity between biases of linear models")
plt.imshow(similarities, cmap=plt.cm.YlGnBu)
plt.colorbar()
plt.show()

## Analysing linear models from different layers

In [ ]:
layers = [9, 10, 11, 12, 13]
with_context = False
dataset_name = "CoNLL2003"
method = "in_context"
method = "average"

linear_models = {} #one model per layer


#load linear layers training results
for layer in layers:
    filtered_results = results_linear[
        (results_linear["layer"] == layer) & 
        (results_linear["dataset_name"] == dataset_name) & 
        (results_linear["with_context"] == with_context) &
        (results_linear["extraction_method"] == method)
        ]    
    for i, row in filtered_results.iterrows():
        path = row["path"]
        #open folder
        files = os.listdir(path)
        ckpts = [f for f in files if f.endswith(".pth")]
        if ckpts:
            #load last checkpoint
            ckpt = torch.load(path / ckpts[-1])
            linear_models[layer] = ckpt
            print(f"loaded {ckpts[-1]} from {path}")
            break
        else:
            print(f"Job {row['hash']} has {len(ckpts)} checkpoints")


### Computing SVD of the learned weights

In [ ]:
N = 10
first_components = {}
last_components = {}

#svd for the learned weights
for layer, linear_model in linear_models.items():
    with torch.no_grad():
        U, S, Vh = torch.linalg.svd(
            linear_model.weight.float(),
            full_matrices=True)
    first_components[layer] = Vh[:N,:].cpu().numpy()
    last_components[layer] = Vh[-N:,:].cpu().numpy()

    # plot of the singular values and a zoom on the first N
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    plt.suptitle(f"SVD of layer {layer}")
    ax[0].plot(S.detach().cpu().numpy())
    ax[0].set_title("Singular values of the learned weights")
    ax[1].plot(S.detach().cpu().numpy()[:N], 'o-')
    ax[1].set_title(f"Singular values of weights (first {N})")
    ax[1].grid()
    ax[2].plot(S.detach().cpu().numpy()[-N:], 'o-')
    ax[2].set_title(f"Singular values of weights (last {N})")
    ax[2].grid()
    #overall title
    plt.show()

### Comparing first and last components of different layers

In [ ]:
#get list from dict values
first_comps = list(first_components.values())

In [ ]:
print(f"indexes of layers are {list(first_components.keys())}")
plot_sims(list(first_components.values()), 0)
plot_sims(list(last_components.values()), -1)

### Angles between subspaces

In [ ]:

#baseline, compute angle between first and last components
print(f"using the first {N} components")
print(f"indexes of layers are {list(first_components.keys())}\n")

print(f"baseline, angles between random {N}-dimensional spaces")
# shape = first_components[0].shape
comp1 = np.random.rand(shape[1],shape[0])
comp2 = np.random.rand(shape[1],shape[0])
angles = subspace_angles(comp1, comp2)
print(f"mean angle between two random subspaces: {np.rad2deg(np.mean(angles)):.2f}°")
print(f"grassmann between two random subspaces: {np.sqrt(np.sum(angles**2)):.2f}\n")

print(f"angles between first component spaces of {len(linear_models)} linear models")
dist = "max"
dist = "mean"
dist = "grassmann"

plot_angles(first_components.values(), first_components.values(), dist=dist)

print(f"angles between last component spaces of {len(linear_models)} linear models")
plot_angles(last_components.values(), last_components.values(), dist=dist)


print(f"angles between first and last component spaces of {len(linear_models)} linear models")
plot_angles(first_components.values(), last_components.values(), dist=dist)

# Misc
## Correlation between reconstructions and representations counts in the Pile

In [ ]:
res = []
counts = {}

In [ ]:
counts_file = f"/home/morand/code/entityrepresentations/counts_{dataset_name}_TestEntities.json"
print(f"loading {counts_file}...")
with open(counts_file) as json_file:
    counts = json.load(json_file)

entity = "Mc Donald's"
entity = "CALGARY"
print(f"Entity {entity} appears {counts[entity]} times in the Pile")

In [ ]:
import requests, time

payload = {
    'corpus': 'v4_piletrain_llama',
    'query_type': 'count',
    'query': 'University of Washington',
}

for item in tqdm(test_dataset):
    entity = item["entity"]
    if entity not in counts:
        payload['query'] = entity
        result = requests.post('https://api.infini-gram.io/', json=payload).json()
        time.sleep(0.001)
        counts[entity] = result['count']

print(counts)



In [ ]:
print(len(counts))
#save counts
import json
with open('counts_CoNLL2003_TestEntities.json', 'w') as f:
    json.dump(counts, f)